# Stage 3: Exploratory Data Analysis (EDA)
**AI/ML Capstone — Phase 1: Machine Learning Pipeline**  
**Dataset:** Clean Ambient Water Monitoring (`data/processed/clean.parquet`, 1,988,629 rows × 30 columns)  
**Objective:** Statistical profiling, physical distribution analysis, collinearity audit, non-linear feature-target relationships, geographic & temporal distribution shifts, monitoring site sampling skew, and shortcut regression quantification.

---

### Key Questions Addressed in Stage 3:
1. **Target Censoring**: How severe is the ceiling spike at `CCME_Values = 100.0`?
2. **Feature Distributions**: How did Stage 2 cleaning reshape chemical parameters, and which features require log transforms?
3. **Collinearity & VIF**: Are Nitrogen and Nitrate severely collinear, and what are their variance inflation factors?
4. **Dominant Predictors**: Which chemistry parameter exerts the strongest monotonic influence on CCME scores?
5. **Non-Linear Dynamics**: Why does Dissolved Oxygen show near-zero linear correlation despite high biological significance?
6. **Geographic Shortcut Risk**: How much target variance can `Country` alone predict without seeing any water chemistry?
7. **Spatial & Temporal Leakage**: Why does monitoring site longevity mandate `GroupKFold` on `Area`?

In [1]:
import os
import sys
import json
from pathlib import Path
from collections import OrderedDict
import numpy as np
import pandas as pd
from tabulate import tabulate

# Configure root directory
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", 35)
pd.set_option("display.width", 1000)

print(f"Working Project Root: {PROJECT_ROOT}")

Working Project Root: /Users/musthyalasadhvik/Desktop/projects/aml-capstone/AML_Capstone


## 1. Load Clean Parquet Dataset & EDA Results
We load the clean ambient dataset (`data/processed/clean.parquet`) and the pre-computed EDA results (`reports/03_eda_results.json`).

In [2]:
from src.eda import load_clean_data, CHEMISTRY_COLS, TARGET_CONT, TARGET_CLASS, META_COLS

df = load_clean_data()
print(f"Clean Ambient Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")

with open(PROJECT_ROOT / "reports" / "03_eda_results.json") as f:
    eda_results = json.load(f)
print("EDA Results JSON loaded successfully!")

Clean Ambient Dataset: 1,988,629 rows × 30 columns
EDA Results JSON loaded successfully!


## 2. Target Distribution & Severe Censoring at 100.0
We examine `CCME_Values` (continuous regression target) and `CCME_WQI` (derived 5-class categorical label).
- Exactly **823,170 rows (41.39%)** are capped at 100.0.
- Class distribution: **Excellent** (41.39%), **Good** (41.10%), **Fair** (10.10%), **Marginal** (6.60%), **Poor** (0.81%).

In [3]:
y = df[TARGET_CONT]
n_100 = (y == 100.0).sum()
pct_100 = 100.0 * n_100 / len(y)

print(f"Target Count: {len(y):,}")
print(f"Rows capped at exactly 100.0: {n_100:,} ({pct_100:.2f}%)")
print(f"Mean: {y.mean():.2f} ± {y.std():.2f}")
print(f"Median: {y.median():.2f}, IQR: {y.quantile(0.75) - y.quantile(0.25):.2f}")

class_summary = pd.DataFrame({
    "Class": df[TARGET_CLASS].value_counts().index,
    "Count": df[TARGET_CLASS].value_counts().values,
    "Percentage": (df[TARGET_CLASS].value_counts(normalize=True) * 100).round(2).values
})
print(tabulate(class_summary, headers="keys", tablefmt="github", showindex=False))
print("[Figure 1 saved to reports/figures/fig01_target_distribution.png]")

Target Count: 1,988,629
Rows capped at exactly 100.0: 823,170 (41.39%)
Mean: 89.62 ± 13.14
Median: 92.87, IQR: 14.09
| Class     |   Count |   Percentage |
|-----------|---------|--------------|
| Excellent |  823170 |        41.39 |
| Good      |  817230 |        41.1  |
| Fair      |  200763 |        10.1  |
| Marginal  |  131327 |         6.6  |
| Poor      |   16139 |         0.81 |
[Figure 1 saved to reports/figures/fig01_target_distribution.png]


![Figure 1: Target Distribution](../reports/figures/fig01_target_distribution.png)

## 3. Feature Distributions Before vs After Cleaning
Comparing the raw dataset (`data/raw/Combined_dataset.csv`) vs the clean ambient dataset (`data/processed/clean.parquet`):
- Non-ambient industrial effluent and sewage discharges (625,265 rows) have been removed.
- Physical impossibilities (pH outside [0, 14], temp > 40°C, zero DO, negative concentrations) have been resolved.
- Skewed concentrations (Ammonia, BOD, Orthophosphate, Nitrogen, Nitrate) are plotted on a log10 scale.

In [4]:
b_a_stats = eda_results["before_after_stats"]
ba_rows = []
for col, s in b_a_stats.items():
    ba_rows.append({
        "Feature": col,
        "Raw Mean": s["raw_mean"],
        "Clean Mean": s["clean_mean"],
        "Raw Median": s["raw_median"],
        "Clean Median": s["clean_median"],
        "Clean Max": s["clean_max"]
    })
print(tabulate(ba_rows, headers="keys", tablefmt="github", showindex=False))
print("[Figure 2 saved to reports/figures/fig02_feature_distributions_before_after.png]")

| Feature                          |   Raw Mean |   Clean Mean |   Raw Median |   Clean Median |   Clean Max |
|----------------------------------|------------|--------------|--------------|----------------|-------------|
| Ammonia (mg/l)                   |     1.1585 |       0.4582 |        0.055 |          0.039 |       200   |
| Biochemical Oxygen Demand (mg/l) |     4.8511 |       3.1492 |        2.7   |          2.1   |       255   |
| Dissolved Oxygen (mg/l)          |    10.0066 |       9.9432 |       10.2   |         10.2   |        20   |
| Orthophosphate (mg/l)            |     0.7093 |       0.3069 |        0.106 |          0.068 |        90   |
| pH (ph units)                    |     7.7357 |       7.7695 |        7.78  |          7.8   |        14   |
| Temperature (cel)                |    11.8413 |      11.5004 |       11.46  |         11.3   |        39.9 |
| Nitrogen (mg/l)                  |     5.2083 |       4.268  |        3.99  |          3.1   |        46   |
|

![Figure 2: Feature Distributions Before vs After Cleaning](../reports/figures/fig02_feature_distributions_before_after.png)

## 4. Collinearity Audit: Pearson, Spearman & Variance Inflation Factors (VIF)
We inspect linear (Pearson) vs monotonic rank (Spearman) correlations:
- **Total Nitrogen vs Nitrate**: $\rho = 0.907$ (Pearson $r = 0.732$). Nitrogen has VIF = 2.27, Nitrate has VIF = 2.16 (the highest among all chemistry features).
- **Orthophosphate**: Exhibits an overwhelming rank correlation of $\rho = -0.913$ with the target `CCME_Values`.

In [5]:
vif_table = pd.DataFrame([
    {"Feature": col, "VIF": eda_results["correlation_analysis"]["vif"][col],
     "Status": "High Collinearity (N-species)" if eda_results["correlation_analysis"]["vif"][col] > 2.0 else "Acceptable"}
    for col in CHEMISTRY_COLS
])
print("### Variance Inflation Factors (VIF):")
print(tabulate(vif_table, headers="keys", tablefmt="github", showindex=False))

print("### Target Correlations (Pearson vs Spearman):")
corr_table = pd.DataFrame([
    {
        "Feature": col,
        "Pearson r": eda_results["correlation_analysis"]["pearson_target_corr"][col],
        "Spearman rho": eda_results["correlation_analysis"]["spearman_target_corr"][col]
    }
    for col in CHEMISTRY_COLS
]).sort_values("Spearman rho")
print(tabulate(corr_table, headers="keys", tablefmt="github", showindex=False))
print("[Figure 3 saved to reports/figures/fig03_correlation_heatmaps.png]")

### Variance Inflation Factors (VIF):
| Feature                          |    VIF | Status                        |
|----------------------------------|--------|-------------------------------|
| Ammonia (mg/l)                   | 1.1185 | Acceptable                    |
| Biochemical Oxygen Demand (mg/l) | 1.1123 | Acceptable                    |
| Dissolved Oxygen (mg/l)          | 1.1239 | Acceptable                    |
| Orthophosphate (mg/l)            | 1.1082 | Acceptable                    |
| pH (ph units)                    | 1.0449 | Acceptable                    |
| Temperature (cel)                | 1.1291 | Acceptable                    |
| Nitrogen (mg/l)                  | 2.5366 | High Collinearity (N-species) |
| Nitrate (mg/l)                   | 2.4238 | High Collinearity (N-species) |
### Target Correlations (Pearson vs Spearman):
| Feature                          |   Pearson r |   Spearman rho |
|----------------------------------|-------------|----------------|

![Figure 3: Correlation Heatmaps](../reports/figures/fig03_correlation_heatmaps.png)

## 5. Non-Linear Feature-Target Relationships
Hexbin density overlays and binned means reveal strong non-linear thresholds:
- **Orthophosphate**: Extremely steep initial cliff: concentrations above 0.1 mg/L cause immediate collapse in water quality.
- **Dissolved Oxygen**: Displays a non-linear biological response ($r \approx -0.002$) because both hypoxia (<5 mg/L) and super-saturation (>14 mg/L) represent impaired aquatic environments.

In [6]:
print("[Figure 4 saved to reports/figures/fig04_feature_target_relationships.png]")

[Figure 4 saved to reports/figures/fig04_feature_target_relationships.png]


![Figure 4: Feature-Target Relationships](../reports/figures/fig04_feature_target_relationships.png)

## 6. Distribution Shifts: Country, Waterbody Type, Year, and Seasonality
Examining geographic and temporal distribution shifts:
- **Country**: USA and Ireland have median CCME of 100.0, whereas England has a median of 89.63.
- **Waterbody Type**: Lakes have mean 96.95, while Drainage canals have mean 73.91.
- **Seasonality**: Water quality reaches a spring peak in March (mean 90.99) and undergoes a pronounced summer depression in July/August (mean 88.10) with elevated variance.

In [7]:
country_df = pd.DataFrame.from_dict(eda_results["country_target_stats"], orient="index")
print("### Target Distribution by Country:")
print(tabulate(country_df.reset_index(), headers=["Country", "Count", "Mean", "Std", "Median"], tablefmt="github", showindex=False))
print("[Figure 5 saved to reports/figures/fig05_target_by_groups_and_time.png]")

### Target Distribution by Country:
| Country   |   Count |   Mean |   Std |   Median |
|-----------|---------|--------|-------|----------|
| Canada    |    3641 |  89.95 | 12.5  |    93.12 |
| China     |   45977 |  96.51 |  4.62 |   100    |
| England   | 1502632 |  86.84 | 13.69 |    89.63 |
| Ireland   |  234990 |  98.11 |  6.41 |   100    |
| USA       |  201389 |  98.92 |  3.24 |   100    |
[Figure 5 saved to reports/figures/fig05_target_by_groups_and_time.png]


![Figure 5: Target by Groups and Time](../reports/figures/fig05_target_by_groups_and_time.png)

## 7. Invalid & Suspect Values Summary
Visualization of all cleaning actions applied in Stage 2 across duplicate removal, non-ambient segregation, historical cutoff filtering, extreme sensor clipping, and outlier tagging.

In [8]:
inv_table = pd.DataFrame([
    {"Rule / Filter": k, "Records Affected": f"{v['count']:,}", "Pipeline Action": v["action"]}
    for k, v in eda_results["invalid_values_summary"].items()
])
print(tabulate(inv_table, headers="keys", tablefmt="github", showindex=False))
print("[Figure 6 saved to reports/figures/fig06_invalid_values_summary.png]")

| Rule / Filter                      |   Records Affected | Pipeline Action   |
|------------------------------------|--------------------|-------------------|
| Exact Duplicates                   |              9,971 | Dropped           |
| Non-Ambient: Effluent              |            601,550 | Segregated        |
| Non-Ambient: Sewage                |             23,777 | Segregated        |
| Pre-1990 Historical Rows           |            204,112 | Dropped           |
| Extreme Temperature (>40°C)        |              7,284 | Set to NaN        |
| Severe Outlier: BOD >100           |             21,242 | Flagged Only      |
| Severe Outlier: Ammonia >50        |              6,658 | Flagged Only      |
| Severe Outlier: Nitrate >100       |              1,009 | Flagged Only      |
| Severe Outlier: Orthophosphate >50 |                302 | Flagged Only      |
| Impossible pH (<0 or >14)          |                505 | Set to NaN        |
| Anoxic / Zero DO (<0.01)           |  

![Figure 6: Invalid and Suspect Values](../reports/figures/fig06_invalid_values_summary.png)

## 8. Site Sampling Distribution & Split Justification
- The clean ambient dataset contains **46,575 distinct monitoring sites (`Area`)**.
- Extreme skew: 25% of sites have only 1 sample, median count is 8, but the largest site contains **49,149 records**.
- **63.80% of sites contain multi-year observations**, and 40.80% span $\ge 5$ years.
- **Critical Cross-Validation Insight**: A standard random K-fold split guarantees temporal and spatial leakage. A `GroupKFold` split grouped strictly by `Area` (or temporal blocked split) is mandatory.

In [9]:
site_res = eda_results["site_analysis"]
print(f"Total Unique Monitoring Sites (Area): {site_res['total_unique_sites']:,}")
print(f"Single-Year Sites: {site_res['single_year_sites']:,} ({site_res['single_year_pct']}%)")
print(f"Multi-Year Sites:  {site_res['multi_year_sites']:,} ({site_res['multi_year_pct']}%)")
print(f"Sites spanning >= 5 years:  {site_res['sites_ge_5_years']:,} ({site_res['sites_ge_5_pct']}%)")
print(f"Sites spanning >= 10 years: {site_res['sites_ge_10_years']:,} ({site_res['sites_ge_10_pct']}%)")
print("[Figure 7 saved to reports/figures/fig07_site_sampling_distribution.png]")

Total Unique Monitoring Sites (Area): 46,575
Single-Year Sites: 16,859 (36.2%)
Multi-Year Sites:  29,716 (63.8%)
Sites spanning >= 5 years:  19,002 (40.8%)
Sites spanning >= 10 years: 12,602 (27.06%)
[Figure 7 saved to reports/figures/fig07_site_sampling_distribution.png]


![Figure 7: Site Sampling Distribution](../reports/figures/fig07_site_sampling_distribution.png)

## 9. Shortcut Regression Analysis
We quantify how well non-chemistry metadata features predict `CCME_Values`:
- `Country` alone explains **14.03% of the target variance ($R^2 = 0.1403$)**.
- `Waterbody Type` alone explains **4.80% of variance ($R^2 = 0.0480$)**.
- `Country + Waterbody + Month` explains **15.64% of variance** without any chemical measurements!
- **Architectural Decision**: Metadata columns (`Country`, `Area`, `Date`) must serve strictly as cross-validation grouping identifiers, NOT model predictors, to avoid learning opportunistic shortcuts.

In [10]:
sc = eda_results["shortcut_analysis"]
sc_rows = [
    {"Feature Set": "Dummy (Mean Baseline)", "R² Score": sc["dummy_mean"]["r2"], "RMSE": sc["dummy_mean"]["rmse"], "MAE": sc["dummy_mean"]["mae"], "Risk Assessment": "Benchmark"},
    {"Feature Set": "Country Alone", "R² Score": sc["country_alone"]["r2"], "RMSE": sc["country_alone"]["rmse"], "MAE": sc["country_alone"]["mae"], "Risk Assessment": "High Shortcut (14.0% Var)"},
    {"Feature Set": "Waterbody Alone", "R² Score": sc["waterbody_alone"]["r2"], "RMSE": sc["waterbody_alone"]["rmse"], "MAE": sc["waterbody_alone"]["mae"], "Risk Assessment": "Moderate Shortcut (4.8% Var)"},
    {"Feature Set": "Country + Waterbody", "R² Score": sc["country_and_waterbody"]["r2"], "RMSE": sc["country_and_waterbody"]["rmse"], "MAE": sc["country_and_waterbody"]["mae"], "Risk Assessment": "High Shortcut (14.8% Var)"},
    {"Feature Set": "Country + Waterbody + Month", "R² Score": sc["country_waterbody_month"]["r2"], "RMSE": sc["country_waterbody_month"]["rmse"], "MAE": sc["country_waterbody_month"]["mae"], "Risk Assessment": "Severe Shortcut (15.6% Var)"},
]
print(tabulate(sc_rows, headers="keys", tablefmt="github", showindex=False))

| Feature Set                 |   R² Score |    RMSE |    MAE | Risk Assessment              |
|-----------------------------|------------|---------|--------|------------------------------|
| Dummy (Mean Baseline)       |     0      | 13.1585 | 9.6606 | Benchmark                    |
| Country Alone               |     0.1403 | 12.2008 | 8.568  | High Shortcut (14.0% Var)    |
| Waterbody Alone             |     0.048  | 12.8386 | 9.397  | Moderate Shortcut (4.8% Var) |
| Country + Waterbody         |     0.1477 | 12.1479 | 8.5089 | High Shortcut (14.8% Var)    |
| Country + Waterbody + Month |     0.1564 | 12.0859 | 8.4558 | Severe Shortcut (15.6% Var)  |


## 10. Summary of 10 Number-Backed Insights & Stage 4 Roadmap
1. **Target Censoring**: 41.39% of ambient records sit at 100.0 $\rightarrow$ post-prediction boundary clipping $[0, 100]$ and tree/censored regressions required.
2. **Dominant Predictor**: Orthophosphate has rank correlation $\rho = -0.913$ $\rightarrow$ logarithmic transform $\log(1 + \text{Orthophosphate})$ and saturation proxies.
3. **Collinearity**: Nitrogen and Nitrate have $\rho = 0.907$ and VIF $\approx 2.2$ $\rightarrow$ ratio feature `Nitrate / Nitrogen` and regularized models.
4. **Non-Linear DO**: Dissolved oxygen has biological two-tailed impairment $\rightarrow$ non-linear $|DO - 10|$ deviation feature.
5. **Geographic Shortcut**: Country predicts $R^2 = 0.1403$ alone $\rightarrow$ quarantine Country as a spatial validation group.
6. **Waterbody Disparity**: Explains $R^2 = 0.0480$ $\rightarrow$ test via ablation study.
7. **Seasonality Cycle**: July/August CCME drops to 88.10 $\rightarrow$ cyclical features $\sin(\text{month}), \cos(\text{month})$.
8. **Site Heterogeneity**: Area observation counts span 1 to 49,149 $\rightarrow$ evaluate sample weighting.
9. **Site Longevity**: 63.80% of sites have multi-year records $\rightarrow$ strictly employ `GroupKFold(groups=Area)`.
10. **Skewness**: Chemistry concentrations span multiple orders of magnitude $\rightarrow$ implement `RobustScaler` / `PowerTransformer` pipeline.